Transformers 라이브러리

인코더 기반 모델
1. klue/bert-base 기본 모델: 파인 튜닝 전 모델 분류(감성분석)
2. klue/bert-base 기본 모델: 파인 튜닝 후 모델 분류(감성분석)
- 학습용 데이터셋 준비
- 데이터셋 전처리(토크나이징)
- 트레이너(Trainer) 설정 
- 학습 실행

In [ ]:
# 인코더 기반 모델
# AutoTokenizer, AutoModelForSequenceClassification
# 일반적인 사전학습 모델은 미세조정이 진행되지 않은 사전학습 모델 가중치를 불러오게 됨
# 이런 경우 미세조정에 사용되는 설정 값이 모델에 포함되어 있지 않기 때문에 추가 설정 값이 필요할 경우가 있음
# 대표 분류 태스크는 분류할 클래스 개수를 뜻하는 num_labels 값을 입력해야 되며, 입력하지 않을시 기본값은 2가 들어가게 됨
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# 모델 이름 지정
model_name = "klue/bert-base"
# 사전 학습 모델 로드
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

# 모델 확인
# (classifier): Linear(in_features=768, out_features=2, bias=True) out_features=2: 분류할 클래스 개수는 2
print(f"model: {model}")

In [ ]:
# model.config.id2label 에서는 해당 모델이 지닌 클래스를 볼 수 있음
# 모델 클래스 확인: {0: 'LABEL_0', 1: 'LABEL_1'} 2개의 클래스 확인
print(f"model.config.id2label: {model.config.id2label}")

In [ ]:
# 인코더 기반 모델
# 1. klue/bert-base 기본 모델: 파인 튜닝 전 모델 분류(감성분석)
# klue/bert-base 순정 모델의 한계: 감성 분석을 하도록 학습된 모델이 아니라, 한국어 문장의 맥락을 이해하도록 사전 학습된 기본 모델
# 무작위 분류기(Classifier) 장착: num_labes=2를 주고 모델을 불러오면, 
# 허깅페이스는 2개 클래스로 분류하고 싶구나 하고 모델 맨 끝에 새로운 분류기 레이어를 무작위(Random) 초기화해서 억지로 붙여준다
# 모델 입장에서는 0번이 무슨 뜻이고 1번이 무슨 뜻인지 전혀 학습된 적이 없기 때문에, 모델 임의로 붙인 LABEL_0/LABEL_1 랜던하게 반환하게 된다
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# 1. 디바이스 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device}")

# 모델 이름 지정
model_name = "klue/bert-base"
# 2. 사전 학습 모델: 감성 분류
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

# 3. 모델 -> 디바이스 이동
model.to(device)

# 4. 평가/추론시 설정: 드롭아웃 비활성화, 정규화 레이어 고정
model.eval()

# 5. 입력 텍스트 준비
inputs_text = "안녕? 내 강아지는 귀여요."

# 6. 토크나이저
inputs = tokenizer(
    inputs_text, # 텍스트가 텍스트를 쪼개고, 숫자로 변환
    return_tensors="pt", # 텐서 형태 변환
    truncation=True, # 글자수 초과시 자르기
    padding=True, # 가장 긴 문장을 기준으로 짧은 문장의 빈자리를 빈 토큰(padding token)으로 채운다
    max_length=512, # 최대 허용 길이 제한
# 토크나이징이 끝나서 만들어진 텐서 데이터(input_ids, attention_mask 등)를 모델이 올라가 있는 동일한 하드웨어로 전송
).to(device)

# 7. 미분 계산 비활성화 상태에서 추론 실행: 미분(역전파) 계산을 막아서 메모리를 아끼고 속도를 높임
with torch.no_grad():
    # 모델의 전체 출력값
    outputs = model(**inputs)
    print(f"outputs: {outputs}")

    # 로짓(raw score) 값 추출
    logits = outputs.logits
    print(f"logits: {logits}")

# 로짓에서 가장 확률이 높은 최종 예측 클래스(인데스) 추출
# torch.argmax(logits, dim=-1): 텐서에서 가장 큰값이 위치한 인덱스(몇번째 칸인지) 찾아 변환
# dim=-1: 다차원 텐서에서 어떤 방향(차원)을 기준으로 최대값을 찾을 것인가 지정, 가장 마지막 차원 의미
# .item(): 텐서 형태 -> int/float 기본형 데이터로 변환 예) 순수한 숫자 1로 변환
pred_class_id = torch.argmax(input=logits, dim=-1).item()
print(f"pred_class_id: {pred_class_id}")

# 필요시 확률값으로 변환 할 수 있음, 최종합이 1
# 첫번째 문장에 대한 모든 클래스(2개) 확률 리스트
probabilites = torch.softmax(input=logits, dim=-1)
print(f"probabilites: {probabilites}")

# probabilites[0]: 첫번째 문장의 전체 확률 목록을 가져온다.
# [pred_class_id]: 그 2개 확률 목록중에서, 이전에 torch.softmax로 구했던 가장 점수가 높았던 최종 예측 클래스 번호(인덱스)에 해당하는 확률값만 집어낸다
# .item(): 그 확률값 텐서를 기본형 float 형태로 꺼내온다
pred_prob = probabilites[0][pred_class_id].item()
print(f"pred_prob: {pred_prob}")

model_class=model.config.id2label[pred_class_id]

# 최종 모델 추론 결과 확인
print(f"입력 텍스트: {inputs_text}")
print(f"예측된 클래스 번호(Label ID): {pred_class_id}")
print(f"예측 확신도(Probability): {pred_prob:.4f}")
print(f"예측 클래스(model_class): {model_class}")

In [5]:
# 인코더 기반 모델
# 2. klue/bert-base 기본 모델: 파인 튜닝 후 모델 분류(감성분석)
# 2-1. 학습용 데이터셋 준비
# - 학습용 데이터셋: 표준 감성 분석 데이터셋 NSMC(Naver Sentiment Movie Corpus) 
# - 데이터를 불러와 토크나이징을 수행하는 작업
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import load_dataset

# 1. 디바이스 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device}")

# 2. 사전학습 모델 및 토크나이저 로드: num_labels=2 로 감성 분류 설정
model_name = "klue/bert-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
model.to(device) # 모델 -> 디바이스(cpu/gpu) 이동

# 3. 네이버 영화 리뷰(NSMC) 감성 분류 데이터셋 로드
# 훈련용(train), 평가용(test) 데이터 일부를 가져온다
dataset = load_dataset(
    path="nsmc",
    split="train[:2000]",
    trust_remote_code=True, # 라이브러리의 최신 보안 정책때문에 외부 데이터셋 스크립트를 실행할때 "이 코드를 신뢰할 수 있는지 확인했다"
) # 실습용으로 2,000개만 샘플링
eval_dataset = load_dataset(
    path="nsmc", 
    split="test[:500]",
    trust_remote_code=True,
) # 평가용으로 500개만 샘플링

print(f"dataset: {dataset}")

device: cpu


/Users/ai/deep-learning/LLM/rag_pdf_qdrant/venv/lib/python3.11/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at klue/bert-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


dataset: Dataset({
    features: ['id', 'document', 'label'],
    num_rows: 2000
})


In [7]:
# 4. 데이터 전처리 함수: 텍스트를 모델이 읽을 수 있는 토큰 ID 변환
def tokenize_function(examples):
    return tokenizer(
        examples["document"],
        padding="max_length",
        truncation=True,
        max_length=128,
    )

# 5. 데이터셋 전체에 토크나이징 일괄 적용
encoded_train_dataset = dataset.map( # train
    function=tokenize_function,
    batched=True,
    batch_size=1000,
)
encoded_eval_dataset = dataset.map( # test
    function=tokenize_function,
    batched=True,
    batch_size=1000,
)

print("데이터 전처리 완료! 학습용 샘플 예시:")
print(encoded_train_dataset[0])

데이터 전처리 완료! 학습용 샘플 예시:
{'id': '9976970', 'document': '아 더빙.. 진짜 짜증나네요 목소리', 'label': 0, 'input_ids': [2, 1376, 831, 2604, 18, 18, 4229, 9801, 2075, 2203, 2182, 4243, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0,

In [9]:
from transformers import Trainer, TrainingArguments

# 6. 학습 하이퍼파라미터 및 설정 정의
training_args = TrainingArguments(
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    num_train_epochs=3,
    eval_strategy="epoch",
    # evaluation_strategy="epoch",
    logging_strategy="steps",
    logging_steps=10,
    logging_dir="./content/logs",
    save_strategy="epoch",
    output_dir="./content/bert_base__sentiment_classification_ckpt",
    report_to="tensorboard",
)

# 7. Trainer 객체 초기화
trainer = Trainer(
    model=model, # 감성 분류 모델
    args=training_args, # 학습 하이퍼파라미터 설정
    train_dataset=encoded_train_dataset, # 훈련용
    eval_dataset=encoded_eval_dataset, # 평가용
    tokenizer=tokenizer # 토크나이저
)

In [ ]:
# 8. 모델 학습(파인튜닝)
print("모델 파인튜닝(학습)을 시작합니다...")
trainer.train()
print("학습이 완료되었습니다!")